# CeramicNet: общий пайплайн

Ветка pipeline объединяет Dataset и аугментации из datasets с общим обучением и оценкой.
**Train 203 фото / 205 bbox, val 51 фото / 52 bbox, seed 42.** Предметы не пересекаются.

Faster R-CNN и адаптер RT-DETR читают данные через настоящий `dataset.StampDataset`.
MMDetection использует штатный CocoDataset, тот же split и аугментации команды.
Выбор для каждой модели находится в `configs/experiments.json`.

Независимого test и проверенных негативов пока нет. Полное обучение включается
явно. Подробности: `detection_pipeline/README.md`.


In [ ]:
from pathlib import Path
import os, sys

PROJECT = Path.cwd()
if not (PROJECT / "dataset").is_dir() and (PROJECT.parent / "dataset").is_dir():
    PROJECT = PROJECT.parent
assert (PROJECT / "dataset").is_dir(), "Укажи PROJECT — корень CeramicNet"
sys.path.insert(0, str(PROJECT))
IMAGES = Path(os.environ.get("CERAMIC_IMAGES", str(PROJECT.parent / "Клеймы")))
SPLITS = PROJECT / "detection_pipeline/splits/v1_seed42"

# В Colab/Kaggle сначала установи зависимости в подходящем окружении:
# %pip install -r {PROJECT / "detection_pipeline/requirements.txt"}
from detection_pipeline.data import read_json
SETTINGS = read_json(PROJECT / "configs/experiments.json")
SEED, EPOCHS, BATCH_SIZE = SETTINGS["seed"], SETTINGS["epochs"], SETTINGS["batch_size"]


## 1. Одно разбиение для всех

Используем уже сохранённые JSON. Не делаем random_split повторно у каждой модели.
В общих JSON category_id=0; адаптер Faster R-CNN переводит его в label=1.
Четыре относительных пути исправлены в новых JSON, оригинал не менялся.

In [ ]:
from detection_pipeline.data import read_json

manifest = read_json(SPLITS / "manifest.json")
assert manifest["seed"] == SEED
train_coco = read_json(SPLITS / "train.json")
val_coco = read_json(SPLITS / "val.json")
assert not ({im["group_id"] for im in train_coco["images"]} &
            {im["group_id"] for im in val_coco["images"]})
assert not ({im["sha256"] for im in train_coco["images"]} &
            {im["sha256"] for im in val_coco["images"]})
for split, stats in manifest["stats"].items():
    print(split, stats)
print("Классы:", train_coco["categories"])
print("Исправлено путей:", len(manifest["paths_changed"]))


## 2. Проверка изображения и рамок

Рамка в COCO — `[x, y, width, height]`, в torchvision — `[x1, y1, x2, y2]`.
В коде преобразование выполняется явно.

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

for split, data in [("train", train_coco), ("val", val_coco)]:
    record = data["images"][0]
    with Image.open(IMAGES / record["file_name"]) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    for annotation in data["annotations"]:
        if annotation["image_id"] == record["id"]:
            x, y, w, h = annotation["bbox"]
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
    print(split, record["file_name"])
    display(image)


## 3. Подключённый класс Dataset и выбор аугментаций

В configs/experiments.json у каждой модели свой профиль: none, light, rotation;
у RT-DETR и RTMDet также native. В light стоят одинаковые преобразования команды,
применяемые до resize. Можно задать свой operations со всеми параметрами.

Модель сама не ищет лучший профиль: сравниваем заранее согласованные опыты на val.
Два Dataset читают разные JSON, а val всегда без случайных изменений.


In [ ]:
from dataset import StampDataset
from detection_pipeline.datasets import make_datasets, seed_everything
from detection_pipeline.experiments import resolve_augmentation

seed_everything(SEED)
profile, operations = resolve_augmentation("fasterrcnn_resnet50_fpn_v2")
train_ds, val_ds = make_datasets(IMAGES, SPLITS, profile=profile, operations=operations)
assert isinstance(train_ds, StampDataset)
image, target = train_ds[0]
print("Используем:", type(train_ds).__name__, "поверх", StampDataset.__name__)
print("Аугментации:", operations)
print("train:", image.shape, target)
print("val:", val_ds[0][0].shape, "без случайных изменений")


## 4. Faster R-CNN ResNet-50 FPN v2

Готовый цикл обучения: COCO pretrained weights, AdamW, лучший checkpoint по val mAP.
Первые настройки — исходная точка, а не найденные оптимальные значения.
Обучение включается явно. Для Colab/Kaggle перенеси весь пакет и фото с теми же
относительными путями; используй GPU.

In [ ]:
RUN_FASTER_RCNN = False
if RUN_FASTER_RCNN:
    from detection_pipeline.train_fasterrcnn import train
    train(IMAGES, SPLITS, PROJECT / f"runs/fasterrcnn_{profile}_seed42")


## 5. RT-DETRv2-S и RTMDet

Используем общие split и выбор аугментаций из configs/experiments.json.
Обёртки регистрируют наши преобразования/класс в официальных библиотеках.
Репозитории и CUDA-зависимости устанавливаются отдельно в окружении владельца модели.
RT-DETRv2-S соответствует ResNet-18. Технический test_dataloader в MMDetection
указывает на val и не является независимым test.


In [ ]:
from detection_pipeline.native_configs import generate

RTDETR_REPO = None  # Path("/absolute/path/RT-DETR")
MMDET_REPO = None   # Path("/absolute/path/mmdetection")
if RTDETR_REPO or MMDET_REPO:
    files = generate(SPLITS, IMAGES, PROJECT / "runs/configs",
                     rtdetr_repo=RTDETR_REPO, mmdet_repo=MMDET_REPO)
    print("\n".join(files))


Запуск из корня CeramicNet в окружении соответствующей библиотеки:

```bash
python -m detection_pipeline.train_native rtdetr --repo /path/RT-DETR \
  --config runs/configs/rtdetrv2_s_light.yml --seed 42 -t /path/coco_checkpoint.pth

python -m detection_pipeline.train_native mmdet --repo /path/mmdetection \
  --config runs/configs/rtmdet_tiny_light.py
```

Для RTMDet-s меняется имя конфига. Происхождение pretrained weights должно быть
записано: по умолчанию у RTMDet ImageNet backbone, у Faster R-CNN COCO detector.
Перед долгим запуском проверь один батч в окружении владельца модели.


## 6. Общая оценка

После обучения экспортируем checkpoint через общий код. Он запускает inference
и преобразует рамки в исходные координаты COCO XYWH. Пустые результаты сохраняются.
Внутренние метки моделей переводятся обратно в общий category_id=0.

```bash
python -m detection_pipeline.predict_native rtdetr --repo /path/RT-DETR \
  --config runs/configs/rtdetrv2_s_light.yml --checkpoint /path/best.pth \
  --images /path/images --annotations detection_pipeline/splits/v1_seed42/val.json \
  --output runs/rtdetr_val --tune-threshold

python -m detection_pipeline.predict_native mmdet --repo /path/mmdetection \
  --config runs/configs/rtmdet_tiny_light.py --checkpoint /path/best.pth \
  --images /path/images --annotations detection_pipeline/splits/v1_seed42/val.json \
  --output runs/rtmdet_val --tune-threshold
```

Выбираем checkpoint по mAP@[.50:.95], дополнительно смотрим AP75 и precision/recall.
Подбор рабочего порога выполняется только на val, например максимум recall при
precision>=90%. Значение confidence само по себе не доказывает правильность рамки.
На независимом test не используем --tune-threshold; переносим выбранный порог в
общий evaluate через --threshold.

Оценщик проверяет хеш разметки и покрытие всех image_id, включая пустые ответы.
Общие score floor=0.001 и максимум 100 bbox. YOLO нужно обучить и проверить на
этом же split; адаптер yolo_prediction уже есть в adapters.py.


In [ ]:
from detection_pipeline.compare import compare

METRICS_FILES = [
    # PROJECT / "runs/fasterrcnn_light_seed42/best_val_metrics.json",
    # PROJECT / "runs/rtmdet_val/metrics.json",
    # PROJECT / "runs/rtdetr_val/metrics.json",
]
if METRICS_FILES:
    rows = compare(METRICS_FILES, PROJECT / "runs/comparison.csv")
    for row in rows:
        print(row)
else:
    print("После обучения добавь пути к файлам общих метрик. Результатов обучения пока нет.")
